<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HuggingFace

In [ ]:
# @title Install HuggingFace Libraries
#####################################

# ⚠️ Accelerator: Gemma-4-E4B in 4-bit quantization requires ~8–9 GB VRAM. Use e.g. A100 (40 GB) or L4 (24 GB)
# ⚠️ Gemma 4 support requires very recent transformers version. Don't just use 'pip install transformers'
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate bitsandbytes
!pip install -q -U peft trl datasets # Required for PEFT tuning
!pip install wandb -q -U
!pip install -U keras huggingface_hub -q

# Imports
# ⚠️ !!! Restart runtime (Runtime → Restart session) and verify version:
import transformers
import torch
from transformers import pipeline, BitsAndBytesConfig
import wandb
from transformers import pipeline, AutoProcessor, Gemma4ForConditionalGeneration, BitsAndBytesConfig, GenerationConfig
print(f'Transformers version: {transformers.__version__}')  # Should show something like 5.6.0.dev0

# Authentification (Colab Secrets or GCP Secrets to store HF access token)
from google.colab import userdata
from huggingface_hub import login
# Huggingface Token from Colab Secrets
# ⚠️ Create token on https://huggingface.co/settings/tokens and add 'Repositories permissions' for model 'google/gemma-4-E4B-it'
hf_token = userdata.get('HF_TOKEN')
# Log in to Hugging Face
login(token=hf_token)

In [2]:
# @title HuggingFace Inference
#####################################

model_id = "mistralai/Mistral-7B-Instruct-v0.3" # https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
#model_id = "deepseek-ai/DeepSeek-V4-Flash"     # https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
#model_id = "google/gemma-4-E2B-it"
#model_id = "google/gemma-4-E4B-it"
#model_id = "google/gemma-4-26B-A4B-it"
#model_id = "google/gemma-4-31B-it"

# https://www.marktechpost.com/2026/04/29/top-10-kv-cache-compression-techniques-for-llm-inference-reducing-memory-overhead-across-eviction-quantization-and-low-rank-methods/?amp

print(f"✅ Loading model: {model_id}")

# 2. Define Quantization Configuration to fit model on GPU
  # ⚠️ BitsAndBytesConfig via quantization_config is now mandatory for HuggingFace Transformers v5+!
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)
# Rule of thumb for memory requirements during quantization:
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)

# 3. Load and Quantize Model
  # Mistral is text-only → "text-generation". AutoTokenizer is used internally
  # (not AutoProcessor like in Gemma that handles new 'mm_token_type_ids' instead of only Tokens, since no multimodal support)
  # Define HuggingFace pipeline with updated 'dtype' and 'quantization_config'. It loads and wires:
    # Neural network weights of Gemma model (16 GB file)
    # Tokenizer that converts text into numbers the model understands
    # Processor handles multimodal inputs (text + images for Gemma 4)
pipe = pipeline(
    "text-generation",                 # ← text-only task for Mistral, "image-text-to-text" for Gemma 4
    model=model_id,                    # Download original model from HF (~30 GB in float32) compressed to local disc (16 GB in bfloat16) as unquantized file 'model.safetensors: 16.0G'
    device_map="auto",                 # GPU placement: weights loaded into VRAM, quantized on-the-fly to 4-bit NF4 format: 'Loading weights: 2130/2130'
    model_kwargs={
        "dtype": torch.bfloat16,
        "quantization_config": quant_config # Weights read from disk, then further quantized to 4-bit  NF4 format (~8–9 GB) during the GPU load.
    }
)

# Mistral uses plain strings for content (not list-of-dicts like Gemma 4)
messages = [
    {"role": "system",  "content": "You are a helpful assistant."}, # gemma: [{"type": "text", "text": "You are a helpful assistant."}]
    {"role": "user",    "content": "What is Mistral AI and what are their main language models?"}
]

outputs = pipe(messages, max_new_tokens=500)
print(outputs[0]["generated_text"][-1]["content"])

✅ Loading model: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing 

Mistral AI is a cutting-edge AI company based in Paris, France, with a team of experienced researchers and engineers. It was founded in 2020 by Adrien Boussard, Théophile Chenusau, and Clement Chelpan (previously from Google, Facebook AI, and DeepMind).

Mistral AI aims to build large-scale artificial general intelligence, focusing on creating AI models that can understand and generate human-like text, speech, and images, as well as solve complex problems.

As of now, Mistral AI has not publicly released any main language models. However, they have made significant strides in the field of AI research and are expected to release language models in the near future. Stay tuned for updates on Mistral AI's progress!


In [ ]:
# @title 1. Prepare Dataset
#####################################

from datasets import load_dataset
import re

"""
Good call to split it up. For the dataset, I'd actually not go with ultrachat_200k
for a first run — it's 200k multi-turn conversations, way overkill and slow on a single GPU with QLoRA. Better picks:

mlabonne/guanaco-llama2-1k — 1k samples, already chat-formatted, finishes in ~10 min on a single GPU. Perfect smoke test.
HuggingFaceH4/ultrachat_200k — only if you want a real run; subset it to a few thousand rows.

I'll go with mlabonne/guanaco-llama2-1k so your first end-to-end run actually completes.
Note it has a text field already wrapped in Llama-2 [INST] format, which happens to match Mistral's template —
convenient, but we'll re-format to messages so SFTTrainer applies Mistral's template properly.
"""

# Guanaco rows look like: "<s>[INST] question [/INST] answer </s><s>[INST] ... "
# We parse them back into role-tagged messages so SFTTrainer can apply Mistral's
# chat template cleanly. Mistral v0.3 supports only user/assistant (no system).
PATTERN = re.compile(r"\[INST\](.*?)\[/INST\](.*?)(?=<s>\[INST\]|</s>|$)", re.DOTALL)

def to_messages(example):
    turns = PATTERN.findall(example["text"])
    messages = []
    for user, assistant in turns:
        messages.append({"role": "user", "content": user.strip()})
        messages.append({"role": "assistant", "content": assistant.strip()})
    return {"messages": messages}

ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
ds = ds.map(to_messages, remove_columns=ds.column_names)
ds = ds.filter(lambda x: len(x["messages"]) >= 2)  # drop any parse failures

split = ds.train_test_split(test_size=0.05, seed=42)
dataset_train = split["train"]
dataset_test = split["test"]

# Save to disk so the training script can load without re-parsing
dataset_train.save_to_disk("./data/train")
dataset_test.save_to_disk("./data/test")

print(f"Train: {len(dataset_train)} | Test: {len(dataset_test)}")
print("Sample:", dataset_train[0]["messages"][:2])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/950 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Train: 950 | Test: 50
Sample: [{'content': 'Please give me the minimum code necessary to create a Phaser.js game within an HTML document. It should be a window of at least 1024x1024 pixels.', 'role': 'user'}, {'content': 'The minimum HTML code required to create a Phaser.js game within an HTML document with a window of at least 1024x1024 pixels would be:\n\n<!DOCTYPE html>\n<html>\n<head>\n    <script src="https://cdn.jsdelivr.net/npm/phaser@5.0.0/dist/phaser.js"></script>\n</head>\n<body>\n    <script>\n        const game = new Phaser.Game({\n            type: Phaser.AUTO,\n            width: 1024,\n            height: 1024,\n            scene: {\n                create: function() {\n                    // your game logic goes here\n                }\n            }\n        });\n    </script>\n</body>\n</html>\n\nThis code includes the Phaser.js library from a CDN and creates an instance of a Phaser.Game with the specified width and height. The create method of the scene is where you

In [ ]:
# @title 2. Load and Quantize Model
#####################################
from transformers import AutoTokenizer, AutoModelForCausalLM


"""
Heads up before the code: the linked Gemma tutorial uses a vision dataset (image + text), but Mistral-7B-Instruct-v0.3 is text-only.
You can't feed it images. You'll need either a text-only dataset (e.g. `HuggingFaceH4/ultrachat_200k`, `mlabonne/guanaco-llama2-1k`)
or to flatten the Gemma dataset to just its text fields. I'll write the code assuming `dataset_train`/`dataset_test` are text-only chat-formatted.

https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora

Main corrections to your Gemma-derived blocks:
- `Gemma4ForConditionalGeneration` → `AutoModelForCausalLM`
- `AutoProcessor` → `AutoTokenizer` (no images)
- Drop the vision-specific `collate_fn`, `skip_prepare_dataset`, and `dataset_text_field=""` — let SFTTrainer tokenize normally
- `processing_class=processor` → `processing_class=tokenizer`
- No `PatchedClippableLinear` monkey-patch needed — Mistral uses standard `nn.Linear`, so PEFT works out of the box

One more thing on the dataset: SFTTrainer auto-applies `tokenizer.apply_chat_template` if your dataset rows look like
`{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}`. Mistral v0.3's chat template uses
`[INST] ... [/INST]` markers and only supports `user`/`assistant` roles (no `system` — fold system content into the first user turn).
If your dataset has a `system` role, you'll get a template error.
"""

# Select model (https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Define Quantization Configuration to fit model on GPU
  # ⚠️ BitsAndBytesConfig via quantization_config is now mandatory for HuggingFace Transformers v5+!
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
  # Mistral 7B: ~14 GB on disk (bfloat16) → ~4 GB on GPU (4-bit quantized) - Much smaller than Gemma 4 E4B (16 GB → 8-9 GB)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)
# Rule of thumb for memory requirements during quantization:
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)

# Load Tokenizer (text-only — no AutoProcessor since Mistral has no vision tower)
# Load Processor for Gemma (lightweight, handles text+image input preparation): 'processor = AutoProcessor.from_pretrained(model_id)'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Mistral has no pad token by default

# Load and Quantize Model (loads 14GB → quantizes to ~4GB on GPU). Gemma: 'Gemma4ForConditionalGeneration.from_pretrained'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=quant_config,
    device_map="auto")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
# @title 3. Setup LoRA Configuration (train only ~1% of parameters)
#####################################

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare for quantized training.
model = prepare_model_for_kbit_training(model)

# Define LoRA config and wrap the full model.
lora_config = LoraConfig(
    r=16,                 # LoRA rank — higher = more capacity but more memory
    lora_alpha=32,        # Scaling factor (usually 2x rank)
    lora_dropout=0.05,    # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
    # Target modules to tune (attention layers + MLP layers for deeper memory changes)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",     # Attention layers
        "gate_proj", "up_proj", "down_proj"        # MLP layers (allows more capacity/ expressivity, but requires more VRAM)
        ],
    # modules_to_save=["lm_head", "embed_tokens"],  # Optional
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [ ]:
# @title 4. Analysis of Mistral Model
#####################################

# Analysis - Print Top level and inner submodules
print("\n ✅ Top-level submodules of the model:\n" + 40*"-")
for name, module in model.named_children():
    print(name, type(module).__name__)

print("\n ✅ Paths of inner submodules:\n" + 40*"-")
for name, module in model.model.named_children():
    print(name, type(module).__name__)

# Analysis - Inspect the actual LoRA matrix shapes
  # Forward pass is base(x) + (lora_B @ lora_A @ x) * (alpha/r). Seeing (16, 4096) and (4096, 16) makes it click why rank matters.
layer = model.base_model.model.model.layers[0].self_attn.q_proj
print("\n ✅ Inspect the actual LoRA matrix shapes:\n" + 40*"-")
print("base_layer:", layer.base_layer.weight.shape)   # (4096, 4096) frozen
print("lora_A:    ", layer.lora_A.default.weight.shape)  # (r, 4096)
print("lora_B:    ", layer.lora_B.default.weight.shape)  # (4096, r)

# Analysis - Quantify what LoRA actually costs
# For Mistral-7B with our config we should see ~0.4–0.5%. Then bump r from 16 → 64 and re-run — trainable params grow linearly with rank,
# and you can feel the memory/quality tradeoff.
"""
**Result:** 1.1% trainable is healthy and exactly what QLoRA is designed to deliver. Breaking down the 41.9M:

- Mistral-7B has 32 layers × 7 target modules (q/k/v/o + gate/up/down) = 224 LoRA-injected linear layers
- Each gets two matrices: `lora_A` of shape `(r, in_features)` and `lora_B` of shape `(out_features, r)`
- Roughly: `224 × r × (in + out)` ≈ 224 × 16 × ~11000 ≈ 39M, plus a bit of overhead → ~42M ✓

The "3.8B total" is interesting too — Mistral-7B has ~7.2B params, but `print_trainable_parameters` counts 4-bit packed weights
as roughly half (since two 4-bit values pack into one uint8), so you see ~3.8B. Total VRAM for those weights is ~4 GB, plus your
42M trainable params in bf16 (~84 MB), plus optimizer states for *only* those trainable params (AdamW = 2× params in fp32 ≈ ~340 MB),
plus activations. That's why QLoRA fits on a single GPU when full fine-tuning would need ~80 GB+. The takeaway: you're training 1%
of the params, using ~5% of the memory of a full fine-tune, and for instruction-following tasks you typically recover 90%+ of full-FT quality.
That's the whole pitch of QLoRA in one ratio.

!! --> Try `r=64` later and you'll see this jump to ~4.4% trainable — useful intuition for the rank/capacity tradeoff.
"""
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("\n ✅ Quantify what LoRA actually costs:\n" + 40*"-")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")


# Analysis - Confirm quantization actually happened
  # The base layer should be a Linear4bit, not a regular nn.Linear.
  # And check VRAM: torch.cuda.memory_allocated() / 1e9 right after model load — should be ~4–5 GB for Mistral-7B in 4-bit vs ~14 GB unquantized.
print("\n ✅ Confirm quantization actually happened:\n" + 40*"-")
print(type(layer.base_layer).__name__)   # Linear4bit
print(layer.base_layer.weight.dtype)     # torch.uint8 (packed 4-bit)

# Print all modules
"""
The contrast with your Gemma 4 dump is the whole point: Mistral has 32 layers (vs Gemma 4's 42), one flat layers ModuleList,
and crucially no vision_tower / audio_tower / embed_vision / embed_audio siblings. That's why target_modules=["q_proj", "k_proj", ...] works
without qualification — every q_proj in the model is a language-model attention projection,
so PEFT can't accidentally attach LoRA adapters to a vision encoder.

One small gotcha: if you run this after get_peft_model(model, lora_config), the structure changes — everything gets wrapped under
base_model.model.* and the Linear layers become lora.Linear. So run this snippet right after AutoModelForCausalLM.from_pretrained(...)
and before prepare_model_for_kbit_training / get_peft_model, otherwise the output will be much noisier and harder to read.
"""
# Mistral is text-only, so there's nothing to exclude for PEFT — but it's still
# useful to confirm the module paths so target_modules in LoraConfig are correct.
print("\n" + 60*"=" + "\n ✅ Print all named modules to see the exact structure:\n" + 60*"=")
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)

# Check which params are actually frozen
  # Sanity check that only LoRA weights have grads. You should see only lora_A, lora_B (and lora_magnitude_vector if DoRA were on, which it isn't here).
  # Everything else — base_layer, embed_tokens, lm_head, layernorms — should be frozen.
print("\n" + 60*"=" + "\n ✅ Check which params are frozen:\n" + 60*"=")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, tuple(p.shape))

# Look at the chat template itself
  # This is the part most people skip and then get confused by. You'll see Mistral's [INST] ... [/INST] markers and the </s> end-of-turn.
  # Understanding this explains why role formatting matters and why a template mismatch silently destroys fine-tuning quality.
print("\n" + 60*"=" + "\n ✅ Print chat template:\n" + 60*"=")
print(tokenizer.chat_template)
print(tokenizer.apply_chat_template(
    [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}],
    tokenize=False,
))

In [ ]:
# @title 5. Run Tuning Job
#####################################

# Connect to wandb for Logging
"""
Watch the loss curve, not just the final number. In your SFTConfig, set report_to="wandb" and eval_strategy="steps",
eval_steps=20. A healthy QLoRA run on guanaco-1k should show train loss dropping from ~2.0 to ~1.2-ish over the epochs,
with eval loss tracking it. If eval loss starts climbing while train loss keeps falling — classic overfitting on a 1k dataset,
and a sign to lower epochs or r.

One thing to watch in the wandb dashboard specifically: the `train/grad_norm` chart. Healthy QLoRA runs sit between 0.5 and 2.0.
If you see spikes above 10, your LR is too high; if it's flatlined near zero, something's frozen that shouldn't be.
"""
from google.colab import userdata
import wandb, os

os.environ["WANDB_API_KEY"] = userdata.get("wandb")
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-qlora"

# Run Tuning Job
from trl import SFTConfig, SFTTrainer
from datasets import load_from_disk

# Load datasets BEFORE constructing the trainer
dataset_train = load_from_disk("./data/train")
dataset_test = load_from_disk("./data/test")

# Training Configuration
sft_config = SFTConfig(
    output_dir="./mistral-finetuned",
    num_train_epochs=4,                # ← 4 is too many for 950 samples. With ~950 train samples and effective batch 16, that's ~60 steps/epoch = ~180 total steps.
                                       #   4 epochs on a tiny dataset starts overfitting hard — eval loss U-turn around epoch 2-3.
    per_device_train_batch_size=4,     # ← A100 has 40/80 GB, bs=1 wastes it - we are using maybe 10 GB of 40/80 GB. With QLoRA + gradient checkpointing + seq_len 2048,
                                       #   bs=4 fits comfortably on 40 GB and bs=8 fits on 80 GB. If you OOM, drop back to 2.
    gradient_accumulation_steps=4,     # effective batch size = 1 × 4 = 4, or 4 x 4 = 16
    learning_rate=5e-5,                # ← better with 2e-4. Earlier set to '5e-5' was to reduce by 4x to stop oscillation in Gemma 4
    lr_scheduler_type="cosine",        # ← gradually decays LR as training progresses
    warmup_ratio=0.03,                 # short warmup avoids early-step LR shock (prevents early instability). Standard for QLoRA. Without warmup, first few steps with cosine can be unstable.
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",             # so wandb gets eval loss curve
    eval_steps=20,
    save_strategy="epoch",
    #save_total_limit=2,                  # ← don't fill disk with checkpoints
    #max_seq_length=2048,               # text-only — cap sequence length for memory --> I get an error message that this isn't supported!
    report_to="wandb",
    run_name="mistral-qlora-guanaco-r16",
    #gradient_checkpointing=True,         # ← trade ~20% speed for ~40% less VRAM (pushes batch size higher). Recomputes activations during backward instead of storing them.

)

# Train
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,         # optional but you have it
    processing_class=tokenizer,        # tokenizer for Mistral; would be `processor` for Gemma (vision)
    # peft_config=lora_config,         # not needed — model already wrapped via get_peft_model()
    # data_collator=collate_fn,        # not needed — text-only, SFTTrainer handles it
)
trainer.train()

# For ~950 training samples we will get 950/16 ≈ 60 steps/epoch × 4 epochs = ~240 steps total- Should finish under 30 min on an A100.
# Save LoRA Adapter (saves only small LoRA weights (~50 MB), not the full 4 GB model)
model.save_pretrained("./mistral-lora-adapter")
tokenizer.save_pretrained("./mistral-lora-adapter")  # for Gemma: 'processor.save_pretrained(...)'

In [ ]:
# @title 6. Connect PEFT Layer to Base Model
#####################################

from peft import PeftModel

# 1. Load base model ('Gemma4ForConditionalGeneration.from_pretrained' for Gemma)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# Injecting PEFT Adapter - Load the PEFT adapter
model = PeftModel.from_pretrained(base_model, "./mistral-lora-adapter")
print("PEFT model loaded successfully!")

In [ ]:
# @title 7. Run Inference on Tuned Model
#####################################

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path = "./mistral-lora-adapter"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=quant_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# Mistral v0.3: no system role — fold any instruction into the first user turn
messages = [
    {"role": "user", "content": "What is Mistral AI and what are their main language models?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,                # ← unpack input_ids + attention_mask
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

# Slice off the prompt tokens so we only print the new response
response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,)
print(response)

https://www.heise.de/hintergrund/So-funktioniert-KV-Cache-Quantisierung-mit-Googles-Verfahren-TurboQuant-11292018.html$0